<a href="https://colab.research.google.com/github/phamquocanh149/SMS_SPAM_Classification/blob/main/SMS_Spam_RNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import numpy as np
import pandas as pd

import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from collections import Counter

In [7]:
nltk.download('stopwords')
nltk.download('punkt_tab')
stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [8]:
path = "https://drive.google.com/uc?id=1qKjiO85UhWBaAQmqT8lyyuq09mXLkiym"

# Read the TSV (tab-separated) file
data = pd.read_csv(path, sep=',', engine='python', encoding='ISO-8859-1', header=None, names=['label', 'message', 'col3', 'col4', 'col5'])

print(data.head())

  label                                            message col3 col4 col5
0    v1                                                 v2  NaN  NaN  NaN
1   ham  Go until jurong point, crazy.. Available only ...  NaN  NaN  NaN
2   ham                      Ok lar... Joking wif u oni...  NaN  NaN  NaN
3  spam  Free entry in 2 a wkly comp to win FA Cup fina...  NaN  NaN  NaN
4   ham  U dun say so early hor... U c already then say...  NaN  NaN  NaN


In [ ]:
print(f"Số cột dữ liệu: {data.columns.tolist()}")

In [9]:
# Drop the unwanted columns (index 2, 3, and 4, which are now named 'col3', 'col4', 'col5')
data = data.drop(['col3', 'col4', 'col5'], axis=1)

print(data.head())

  label                                            message
0    v1                                                 v2
1   ham  Go until jurong point, crazy.. Available only ...
2   ham                      Ok lar... Joking wif u oni...
3  spam  Free entry in 2 a wkly comp to win FA Cup fina...
4   ham  U dun say so early hor... U c already then say...


In [10]:
print(f"Số dòng: {len(data)}")

Số dòng: 5573


In [11]:
import torch


In [12]:
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

In [13]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder

In [14]:
#hàm tiền xử lí chuỗi đầu vào
def preprocess_text(text):
  text = str(text)
  text = text.lower()
  text = re.sub(r"http\S+|www.\S+", '', text) # thay URL = ''
  text = re.sub(r"[^\w\s]", '', text) # loai bo bất kỳ ký tự nào KHÔNG phải là chữ, số, dấu gạch dưới hoặc khoảng trắng.
  text = re.sub(r"\s+", ' ', text).strip() # chuan hoa khoang trang
  word = [word for word in text.split() if word not in stop_words]
  return ' '.join(word)

In [15]:
data['message'] = data['message'].apply(preprocess_text)

In [16]:
X = data['message'][1:].tolist()
y = data['label'][1:].tolist()

In [18]:
print(y[:5])

['ham', 'ham', 'spam', 'ham', 'ham']


In [19]:
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)

In [20]:
print(y[:5])

[0 0 1 0 0]


In [37]:
print(X[:5])

['go jurong point crazy available bugis n great world la e buffet cine got amore wat', 'ok lar joking wif u oni', 'free entry 2 wkly comp win fa cup final tkts 21st may 2005 text fa 87121 receive entry questionstd txt ratetcs apply 08452810075over18s', 'u dun say early hor u c already say', 'nah dont think goes usf lives around though']


In [22]:
#split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=14)

In [23]:
leng = [len(s.split()) for s in X_train]
max_len = max(leng)
print(max_len)

72


In [24]:
token = [word_tokenize(sen) for sen in X_train]

In [25]:
word_count = Counter([word for sen in token for word in sen])

In [26]:
#build vocabulary
voca = {word: i+2 for i, word in enumerate(word_count.items())}
voca['<pad>'] = 0
voca['<unk>'] = 1

In [27]:
def encode_sen(sen, voca, max_length = max_len):
  text = word_tokenize(sen)
  encode = [voca.get(word, voca['<unk>']) for word in text]
  if len(encode) < max_length:
    encode += [voca['<pad>']] * (max_length - len(encode))
  else:
    encode = encode[:max_length]
  return encode

In [28]:
encode_X_train = [encode_sen(sen, voca) for sen in X_train]
encode_X_test = [encode_sen(sen, voca) for sen in X_test]

In [29]:
print(encode_X_train[:5])

[[1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

In [30]:
class TextDataset(Dataset):
  def __init__(self, X, y):
    self.X = torch.tensor(X, dtype=torch.long)
    self.y = torch.tensor(y, dtype=torch.long)

  def __getitem__(self, idx):
    return self.X[idx], self.y[idx]

  def __len__(self):
    return len(self.X)

In [31]:
train = TextDataset(encode_X_train, np.array(y_train, dtype=int))
test = TextDataset(encode_X_test, np.array(y_test, dtype=int))

In [32]:
train_loader = DataLoader(train, batch_size=4, shuffle=True)
test_loader = DataLoader(test, batch_size=4, shuffle=False)

In [33]:
class RNNClassifier (nn.Module):
  def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, num_layers = 2):
    super(RNNClassifier, self).__init__()
    self.embedding = nn.Embedding(vocab_size, embedding_dim)
    self.rnn = nn.RNN(embedding_dim, hidden_dim, num_layers, batch_first=True)
    self.fc = nn.Linear(hidden_dim, output_dim)
  def forward(self, x):
    embedded = self.embedding(x)
    output, hidden = self.rnn(embedded)
    # Get the hidden state of the last layer for all batches
    last_hidden = hidden[-1]
    logits = self.fc(last_hidden)
    return logits
  def init_hidden(self, batch_size, device): #khoi tao hidden dau tien
    return torch.zeros(1, batch_size, self.rnn.hidden_size).to(device)

In [34]:
def train_with_logging(model, data_loader, optimizer, criterion, seq_len, epochs, device):
    model.train()
    losses = []
    best_loss = float('inf')   # Khởi tạo loss tốt nhất
    best_model_state = None    # Để lưu lại trạng thái tốt nhất

    for epoch in range(epochs):
        total_loss = 0
        for batch in data_loader:
            inputs, targets = batch
            inputs, targets = inputs.to(device), targets.to(device)

            batch_size = inputs.size(0)

            optimizer.zero_grad()
            logits = model(inputs)
            loss = criterion(logits, targets)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5)
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(data_loader)
        losses.append(avg_loss)

        # Kiểm tra và lưu lại best loss
        if avg_loss < best_loss:
            best_loss = avg_loss
            best_model_state = model.state_dict()
            print(f"Epoch {epoch+1}, Loss: {avg_loss:.4f} (best so far)")
        else:
            print(f"Epoch {epoch+1}, Loss: {avg_loss:.4f}")

    if best_model_state is not None:
        model.load_state_dict(best_model_state)

    return losses, best_loss


In [35]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = RNNClassifier(
    vocab_size=len(voca),
    embedding_dim=256,
    hidden_dim=256,
    output_dim=2
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.00005)


train_with_logging(model, train_loader, optimizer, criterion, seq_len=50, epochs=10, device=device)

Epoch 1, Loss: 0.4034 (best so far)
Epoch 2, Loss: 0.4025 (best so far)
Epoch 3, Loss: 0.4016 (best so far)
Epoch 4, Loss: 0.4011 (best so far)
Epoch 5, Loss: 0.4005 (best so far)
Epoch 6, Loss: 0.4012
Epoch 7, Loss: 0.4014
Epoch 8, Loss: 0.4003 (best so far)
Epoch 9, Loss: 0.4010
Epoch 10, Loss: 0.4013


([0.40337406819844995,
  0.4025091600391362,
  0.4016303580839003,
  0.4011253085839374,
  0.4005185662497319,
  0.4011594345417258,
  0.4013634675286811,
  0.4003321783391617,
  0.40101806762640785,
  0.4012869074272468],
 0.4003321783391617)

In [36]:
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        inputs, labels = batch
        inputs, labels = inputs.to(device), labels.to(device)

        logits= model(inputs)
        preds = torch.argmax(logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())


cm = confusion_matrix(all_labels, all_preds)
class_names = label_encoder.classes_.astype(str)


print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=label_encoder.classes_.astype(str)))


Classification Report:
              precision    recall  f1-score   support

         ham       0.88      1.00      0.94       979
        spam       0.00      0.00      0.00       136

    accuracy                           0.88      1115
   macro avg       0.44      0.50      0.47      1115
weighted avg       0.77      0.88      0.82      1115



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
